# Notebook 01: Train + Generate with EngiOpt CGAN-2D (DCC26)


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'sqlitedict', 'torch', 'torchvision', 'matplotlib', 'pandas', 'tqdm', 'tyro', 'wandb']
ENGIOPT_GIT = 'git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt'

if IN_COLAB or FORCE_INSTALL:
    print('Installing base dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Installing EngiOpt from GitHub branch...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', ENGIOPT_GIT])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from engibench.problems.beams2d.v0 import Beams2D

try:
    from engiopt.cgan_2d.cgan_2d import Generator as EngiOptCGAN2DGenerator
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Could not import engiopt model class. Run the bootstrap cell first; on Colab, restart runtime after install if needed.'
    ) from exc


# Optional W&B artifact flow (disabled by default)
USE_WANDB_ARTIFACTS = False
WANDB_PROJECT = 'dcc26-workshop'
WANDB_ENTITY = None
WANDB_ARTIFACT_NAME = 'dcc26_beams2d_generated_artifacts'
WANDB_ARTIFACT_ALIAS = 'latest'


def resolve_artifact_dir(create: bool = False) -> Path:
    in_colab = 'google.colab' in sys.modules
    if in_colab:
        path = Path('/content/dcc26_artifacts')
    else:
        path = Path('workshops/dcc26/artifacts')

    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path


SEED = 7
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if th.cuda.is_available():
    th.cuda.manual_seed_all(SEED)

DEVICE = th.device('cuda' if th.cuda.is_available() else 'cpu')
print('device:', DEVICE)

ARTIFACT_DIR = resolve_artifact_dir(create=True)
print('artifact dir:', ARTIFACT_DIR)

CKPT_PATH = ARTIFACT_DIR / 'engiopt_cgan2d_generator_supervised.pt'
LATENT_DIM = 32


In [ ]:
problem = Beams2D(seed=SEED)
train_ds = problem.dataset['train']
test_ds = problem.dataset['test']

condition_keys = problem.conditions_keys
print('condition keys:', condition_keys)

# Build compact train subset to keep runtime stable in workshop
N_TRAIN = 512
subset_idx = np.random.default_rng(SEED).choice(len(train_ds), size=N_TRAIN, replace=False)

conds_np = np.stack([np.array(train_ds[k])[subset_idx].astype(np.float32) for k in condition_keys], axis=1)
designs_np = np.array(train_ds['optimal_design'])[subset_idx].astype(np.float32)

# EngiOpt CGAN generator emits tanh-scaled outputs in [-1, 1]
targets_np = (designs_np * 2.0) - 1.0

print('conditions shape:', conds_np.shape)
print('designs shape:', designs_np.shape)
print('target range:', float(targets_np.min()), 'to', float(targets_np.max()))


In [ ]:
model = EngiOptCGAN2DGenerator(
    latent_dim=LATENT_DIM,
    n_conds=conds_np.shape[1],
    design_shape=problem.design_space.shape,
).to(DEVICE)

optimizer = th.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()


def sample_noise(batch_size: int) -> th.Tensor:
    return th.randn((batch_size, LATENT_DIM), device=DEVICE, dtype=th.float32)


In [ ]:
TRAIN_FROM_SCRATCH = True
EPOCHS = 8
BATCH_SIZE = 64

if TRAIN_FROM_SCRATCH:
    ds = TensorDataset(th.tensor(conds_np), th.tensor(targets_np))
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for cond_batch, target_batch in dl:
            cond_batch = cond_batch.to(DEVICE)
            target_batch = target_batch.to(DEVICE)

            pred = model(sample_noise(cond_batch.shape[0]), cond_batch)
            loss = criterion(pred, target_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item())

        print(f'epoch {epoch + 1:02d}/{EPOCHS} - loss: {epoch_loss / len(dl):.4f}')

    th.save(
        {
            'model': model.state_dict(),
            'condition_keys': condition_keys,
            'latent_dim': LATENT_DIM,
            'model_family': 'engiopt.cgan_2d.Generator',
        },
        CKPT_PATH,
    )
    print('saved checkpoint to', CKPT_PATH)
elif CKPT_PATH.exists():
    ckpt = th.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print('loaded checkpoint from', CKPT_PATH)
else:
    raise FileNotFoundError(f'No checkpoint found at {CKPT_PATH}. Set TRAIN_FROM_SCRATCH=True or provide a checkpoint.')


In [ ]:
rng = np.random.default_rng(SEED)
N_SAMPLES = 24
selected = rng.choice(len(test_ds), size=N_SAMPLES, replace=False)

test_conds = np.stack([np.array(test_ds[k])[selected].astype(np.float32) for k in condition_keys], axis=1)
baseline_designs = np.array(test_ds['optimal_design'])[selected].astype(np.float32)

model.eval()
with th.no_grad():
    tanh_out = model(sample_noise(N_SAMPLES), th.tensor(test_conds, device=DEVICE))
    gen_designs_t = ((tanh_out.clamp(-1.0, 1.0) + 1.0) / 2.0).clamp(0.0, 1.0)
gen_designs = gen_designs_t.detach().cpu().numpy().astype(np.float32)

print('generated shape:', gen_designs.shape)
print('baseline shape:', baseline_designs.shape)


In [ ]:
conditions_records = []
for i in range(N_SAMPLES):
    rec = {}
    for j, k in enumerate(condition_keys):
        v = test_conds[i, j]
        rec[k] = bool(v) if k == 'overhang_constraint' else float(v)
    conditions_records.append(rec)

generated_path = ARTIFACT_DIR / 'generated_designs.npy'
baseline_path = ARTIFACT_DIR / 'baseline_designs.npy'
conditions_path = ARTIFACT_DIR / 'conditions.json'

np.save(generated_path, gen_designs)
np.save(baseline_path, baseline_designs)
with open(conditions_path, 'w', encoding='utf-8') as f:
    json.dump(conditions_records, f, indent=2)

print('Saved artifacts to', ARTIFACT_DIR)

if USE_WANDB_ARTIFACTS:
    try:
        import wandb

        run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type='artifact-upload', reinit=True)
        artifact = wandb.Artifact(WANDB_ARTIFACT_NAME, type='dataset', description='DCC26 Notebook 01 generated artifacts')
        artifact.add_file(str(generated_path))
        artifact.add_file(str(baseline_path))
        artifact.add_file(str(conditions_path))
        run.log_artifact(artifact, aliases=[WANDB_ARTIFACT_ALIAS])
        run.finish()
        print('Uploaded artifacts to W&B:', WANDB_ARTIFACT_NAME)
    except Exception as exc:
        print('W&B upload failed (continuing with local artifacts only):', exc)


In [ ]:
# Quick visual check of generated designs
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(gen_designs[i], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(f'gen {i}')
plt.tight_layout()
plt.show()

## Next

Continue with **Notebook 02** to validate and evaluate generated designs against baselines.
